# 00 — The Simplest Event Loop: Callbacks

**Goal:** Build a minimal event loop. Step 1 of building asyncio from scratch.

## What IS an event loop?

```python
while has_work:
    run_ready_callbacks()
    check_timers()
    poll_io()
```

That's it. Everything else is details.

This notebook: callback queue + `call_soon` + `call_later`.

## Step 1 — Build EventLoop

Build an `EventLoop` class with:
- `self._ready = deque()` — callbacks ready to run NOW, each entry is `(callback, args)`
- `self._scheduled = []` — a min-heap of `(when, seq, callback, args)` for timers
- `self._seq = 0` — for heap tie-breaking
- `call_soon(callback, *args)` — append to ready queue
- `call_later(delay, callback, *args)` — push onto scheduled heap with `when = time.time() + delay`
- `stop()` — sets a flag to exit the loop
- `run_forever()` — loop calling `_run_once()` until stopped
- `_run_once()` — move due timers to ready, run all ready callbacks, if nothing ready sleep until next timer

Test with a countdown that schedules itself recursively:
```python
def countdown(n):
    if n > 0:
        print(f"Countdown: {n}")
        loop.call_later(1.0, countdown, n - 1)
    else:
        print("BLASTOFF!")
        loop.stop()
```

In [ ]:
from collections import deque
import heapq
import time

# Step 1 — Build EventLoop


### Question 0.1
The countdown doesn't block for 1 second — it schedules ITSELF to run 1 second later and returns immediately. Instead of `time.sleep(1)`, you say "call me back in 1 second".

**This is how Node.js works.** But callbacks are ugly — that's why Python added coroutine support.

*Your answer:*


## Step 2 — Multiple concurrent countdowns

Launch 3 countdowns with different durations (3, 5, 2). Track active count. Stop when all done.

Should complete in ~5 seconds (max), not 10 (sum).

In [ ]:
# Step 2 — Concurrent countdowns


### Question 0.2
Callbacks work but are ugly. You have to manually track state and chain callbacks. Compare:

**Callback hell:**
```python
def step1():
    do_something()
    loop.call_later(1, step2)
```

**Coroutine (what we'll build next):**
```python
async def task():
    do_something()
    await sleep(1)    # looks sequential!
    do_more()
```

**Next notebook:** Add coroutine support to your event loop.